In [ ]:
from collections.abc import Callable, Sequence
from pathlib import Path
import pandas as pd
from torch import nn
from model.cnn import CNN

from dataset.viz.ball_localization_visualizer import BallLocalizationVisualizer
from model.engine.experiment_config import ExperimentConfig
from model.engine.ball_data_module import BallDataModule

from model.engine.ball_localization_trainer import BallLocalizationTrainer
from model.engine.model_checkpoint import ModelCheckpoint
from model.eval.ball_location_evaluator import BallLocalizationEvaluator


In [ ]:
config = ExperimentConfig(
    raw_data_dir=Path(
        "/home/bugslayer/Documents/research/data/VolleyVision/VballNet/train"
    ),
    processed_data_dir=Path(
        "/home/bugslayer/Documents/research/data/VolleyVision/processed/512x288"
    ),
    checkpoint_path=Path(
        "/home/bugslayer/Documents/research/VolleyVision/notebooks/dataset/ball_model.pth"
    ),
)

data_module = BallDataModule(config).prepare_data()
manifest_df = data_module.manifest_df

train_df = data_module.frame_table("train")
val_df = data_module.frame_table("val")
test_df = data_module.frame_table("test")

train_dataset = data_module.dataset("train")
val_dataset = data_module.dataset("val")
test_dataset = data_module.dataset("test")

train_loader = data_module.dataloader("train")
val_loader = data_module.dataloader("val")
test_loader = data_module.dataloader("test")

display(data_module.split_summary())

In [ ]:
batch = next(iter(train_loader))

print("image:", tuple(batch["image"].shape))
print("target:", tuple(batch["target"].shape))
print("visibility:", tuple(batch["visibility"].shape))

In [ ]:
visualizer = BallLocalizationVisualizer()
visualizer.plot_visible_target(
    train_dataset,
    train_df,
    visible_rank=min(10, int(train_df["visibility"].sum()) - 1),
)

In [ ]:
# Swap this factory to train and evaluate another compatible model.
model_factory: Callable[[], nn.Module] = CNN

device = config.resolved_device()
model = model_factory()
evaluator = BallLocalizationEvaluator(config.thresholds)
trainer = BallLocalizationTrainer(
    model=model,
    evaluator=evaluator,
    device=device,
    learning_rate=config.learning_rate,
)
checkpoint = ModelCheckpoint(config.checkpoint_path)

print(f"Using device: {device}")

In [ ]:
model, history = trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=config.epochs,
    validation_interval=config.validation_interval,
)

display(history.tail())

In [ ]:
saved_checkpoint = checkpoint.save(model)
print(f"Saved checkpoint to {saved_checkpoint}")

## Held-out test results

`BallLocalizationEvaluator` uses the same decoding and metric accumulation for validation and testing. Metrics are aggregated per frame (rather than averaging per-batch medians), and prediction rows remain available for qualitative inspection.

In [ ]:
split_summary = data_module.split_summary()
display(split_summary)

In [ ]:
test_model = checkpoint.load(model_factory(), device)
test_model.eval()

In [ ]:
test_metrics_dict, test_predictions_df = evaluator.evaluate(
    model=test_model,
    data_loader=test_loader,
    device=device,
    frame_table=test_df,
    description="Testing",
)
test_metrics = pd.Series(test_metrics_dict, name="value")
worst_predictions = (
    test_predictions_df
    .dropna(subset=["distance_px"])
    .nlargest(10, "distance_px")
)

display(test_metrics.to_frame())
display(
    worst_predictions[
        [
            "video_id",
            "frame",
            "target_x",
            "target_y",
            "predicted_x",
            "predicted_y",
            "distance_px",
            "peak_value",
        ]
    ]
)

### Qualitative error inspection

The reusable visualizer plots the largest visible-ball errors. Green circles are ground truth and yellow crosses are model predictions.

In [ ]:
visualizer.plot_errors(
    dataset=test_dataset,
    predictions=test_predictions_df,
    count=6,
    columns=3,
)